# Runoff contribution


In [1]:
# general
from pathlib import Path
import pandas as pd
import geopandas as gpd
import xarray as xr
import yaml 
import os

# plotting
import plotly.graph_objects as go
from   plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes/')
os.chdir(wd)

# import config and local module
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

from src.utils import weighted_mean_by_lat

## Data

In [108]:
common_period = slice("2001-01-01", "2024-12-31")

# plot elements
main_basins = gpd.read_file(cfg["dir"]["data"] + "/HYDROATLAS/hybas_sa_lev08_v1c_dissolved.shp")
main_basins["geometry"] = main_basins["geometry"].simplify(0.02, preserve_topology=True)

# polygons / shapes
hydroatlas_shape = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index("gauge_id")
#hydroatlas_shape = hydroatlas_shape[hydroatlas_shape.glacier_area_RGI60 > 0]
hydroatlas_shape['basin_cenlat_rounded'] = hydroatlas_shape['basin_cenlat'].round(0)

# total runoff in mm/day
hydroatlas_runoff = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_hist.nc", chunks = "auto").q_mm_sim
hydroatlas_runoff = hydroatlas_runoff.sel(date=common_period).rename("total_runoff")

# glacier melt in m3/s -> mm/day
hydroatlas_gmelt = xr.open_dataset("data/glacier/run_output_historical.nc").glacier_melt
hydroatlas_gmelt = hydroatlas_gmelt.reindex(gauge_id=hydroatlas_shape.index, fill_value=0)
hydroatlas_gmelt = hydroatlas_gmelt.sel(date=common_period)
hydroatlas_gmelt = (hydroatlas_gmelt * 1000 * 86400) / (hydroatlas_shape.basin_area.values * 1e6)[:, None]

# runoff 
ds = xr.merge([hydroatlas_runoff, hydroatlas_gmelt]).resample(date="MS").sum().load()
ds["gmelt_ratio"] = ds.glacier_melt / ds.total_runoff

ds_annual = ds.resample(date="YS").sum().mean(dim="date")
ds_annual["gmelt_ratio"] = ds_annual.glacier_melt / ds_annual.total_runoff # rewrite

ds_summer = ds.sel(date=ds.date.dt.month.isin([12, 1, 2])).resample(date="YS").sum().mean(dim = "date")
ds_summer["gmelt_ratio"] = ds_summer.glacier_melt / ds_summer.total_runoff # rewrite

ds_monthly = ds["gmelt_ratio"].groupby("date.month").mean()
ds_monthly = ds_monthly.isel(month=ds_monthly.argmax(dim="month"))

In [109]:
# lat processing
hydroatlas_shape = hydroatlas_shape.join([
    ds_annual.total_runoff.rename("runoff_annual").to_pandas(),
    ds_annual.glacier_melt.rename("gmelt_annual").to_pandas(),
    ds_annual.gmelt_ratio.rename("gmelt_ratio_annual").to_pandas(),
    ds_summer.gmelt_ratio.rename("gmelt_ratio_summer").to_pandas(),
    ds_monthly.rename("gmelt_ratio_monthly").to_pandas(),
])

hydroatlas_shape = hydroatlas_shape.sort_values('runoff_annual')
hydroatlas_shape_lat = weighted_mean_by_lat(hydroatlas_shape) 

## Plot (MS)

In [110]:
# Bivariate palette (4x4): cols = runoff (low→high), rows = gmelt (low→high)
n_classes = 4
bivariate_colors = [
    ["#e8e8e8", "#c8e0e0", "#97caca", "#5ac8c8"],  # low gmelt
    ["#e4c5df", "#c4bdd8", "#93adc9", "#5698b9"],  # mid-low gmelt
    ["#d09bc9", "#b094c0", "#8484ae", "#3f6f9f"],  # mid-high gmelt
    ["#be64ac", "#9e5ca0", "#6e4d8e", "#3b4994"],  # high gmelt
]

runoff_bins = pd.qcut(hydroatlas_shape['runoff_annual'], n_classes, labels=False, duplicates='drop')
gmelt_bins  = pd.qcut(hydroatlas_shape['gmelt_annual'],  n_classes, labels=False, duplicates='drop')
point_colors = [bivariate_colors[int(g)][int(r)] for g, r in zip(gmelt_bins, runoff_bins)]

In [135]:
# Create subplot structure
fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True,
    horizontal_spacing=0.08,
    column_widths=[0.6, 0.4],
    specs=[[{"type": "scattergeo"}, {"type": "xy"}]]
)

# Plot Nº1: Bivariate map
fig.add_trace(
    go.Scattergeo(
        lon=hydroatlas_shape['basin_cenlon'],
        lat=hydroatlas_shape['basin_cenlat'],
        mode='markers',
        marker=dict(
            size=6,
            color=point_colors,
            opacity=0.9,
            line=dict(width=0.1, color='white')
        ),
        showlegend=False
    ),
    row=1, col=1
)

# Add river labels
fig.add_trace(go.Choropleth(geojson=eval(main_basins['geometry'].to_json()), locations=main_basins.index, 
                            z=main_basins.NEXT_DOWN, colorscale=["rgba(255,255,255,0.0001)", "rgba(255,255,255,0.0001)"], 
                            marker_line_color = "rgb(210, 226, 230)", showscale=False, marker_line_width=0.3), 
                row=1, col=1)

fig.add_trace(
    go.Scattergeo(
        lat=[-50.0, -47.0, -42.5, -39.0, -37.0, -33.5, -36.0, -30.0, -18.0, -9.0],
        lon=[-76.0, -76.0, -74.0, -69.0, -74.5, -73.0, -67.5, -72.5, -67.0, -76.0],
        text=["Santa<br>Cruz", "Baker", "Puelo", "Negro", "Biobío", "Maipo", "Mendoza", "Elqui", "Titicaca", "Santa"],
        mode="text",
        textfont=dict(size=10, color="#5e8588"),
        showlegend=False
    ), row=1, col=1)


# Update geography settings
water = "#ecf1f8"
fig.update_geos(dict(
    showframe=True, 
    framecolor="#ecf1f8", bgcolor="rgba(210, 226, 230, 0.7)",
    landcolor="white", showland=True, resolution=50,
    showlakes=True, showocean=True, lakecolor=water, oceancolor=water,
    showcoastlines=True, coastlinecolor="rgba(0,0,0,0)",
    showcountries=True, countrycolor="rgba(0,0,0,0.6)", countrywidth=0.1,
    lonaxis_range=[-87, -62], lataxis_range=[-56, -7],
), row=1, col=1)

# Add vertical line on the left
fig.add_shape(type="line", x0=0.06, x1=0.06, y0=0.005, y1=0.995,
              xref="paper", yref="paper", line=dict(width=1, color="#83888f"))

# Add horizontal lines at region boundaries
for boundary_lat in [-7.2, -20, -30, -35, -45, -52, -55.8]:
    y_pos = (boundary_lat - (-56)) / ((-7) - (-56))
    fig.add_shape(type="line", x0=0.055, x1=0.065, y0=y_pos, y1=y_pos,
                  xref="paper", yref="paper", line=dict(width=1, color="#83888f"))

# Add region annotations
for region, (lat_min, lat_max) in cfg["regions"].items():
    mid_lat = (lat_min + lat_max) / 2
    y_pos = (mid_lat - (-56)) / ((-7) - (-56))
    fig.add_annotation(
        text=region.replace(' ', '<br>'), bgcolor=water, borderpad=1,
        x=0.06, y=y_pos, xref="paper", yref="paper",
        showarrow=False, font=dict(size=12, family="Arial"),
        xanchor="center", yanchor="middle",
    )

# Bivariate legend (4x4 grid)
lx0, ly0 = 0.15, 0.60
cell_w, cell_h = 0.035, 0.035
for r in range(n_classes):      # runoff: 0=low → 3=high (left to right)
    for g in range(n_classes):  # gmelt:  0=low → 3=high (bottom to top)
        fig.add_shape(
            type="rect",
            x0=lx0 + r * cell_w,       x1=lx0 + (r + 1) * cell_w,
            y0=ly0 + g * cell_h,        y1=ly0 + (g + 1) * cell_h,
            xref="paper", yref="paper",
            fillcolor=bivariate_colors[g][r],
            line=dict(width=0.5, color="white"),
        )

fig.add_annotation(text="Total runoff →", x=lx0 + 1.8 * cell_w, y=ly0 - 0.012,
                   xref="paper", yref="paper", showarrow=False,
                   font=dict(size=11, family="Arial"), xanchor="center")
fig.add_annotation(text="Glacier melt →", x=lx0 - 0.012, y=ly0 + 1.8 * cell_h,
                   xref="paper", yref="paper", showarrow=False,
                   font=dict(size=11, family="Arial"), xanchor="center", textangle=-90)



# Plot Nº2: Glacier melt to runoff ratio by latitude
fig.add_trace(
    go.Bar(
        x=hydroatlas_shape_lat['gmelt_ratio_monthly']*100,
        y=hydroatlas_shape_lat.index,
        name="Peak month",
        marker_color="#b9b5d4",
        orientation='h',
        opacity=1,
    ),
    row=1, col=2
)

fig.add_trace(
    go.Bar(
        x=hydroatlas_shape_lat['gmelt_ratio_summer']*100,
        y=hydroatlas_shape_lat.index,
        name="Summer",
        marker_color="#be74af",
        orientation='h',
        opacity=1,
    ), 
    row=1, col=2
)

fig.add_trace(
    go.Bar(
        x=hydroatlas_shape_lat['gmelt_ratio_annual']*100,
        y=hydroatlas_shape_lat.index,
        name="Annual",
        marker_color= "#457bb0",
        orientation='h',
        opacity=1,
    ),
    row=1, col=2
)


fig.update_yaxes(title_standoff=2, ticksuffix="ºS", dtick = 5, range=(-56, -7), row=1, col=2)
fig.update_xaxes(title="Contribution rel. runoff(%)", title_standoff = 2,  range = (0, 60), row=1, col=2)

# Update axes + annotations
fig.update_xaxes(ticks="outside", griddash="dot", zeroline=False)
fig.update_yaxes(ticks="outside", griddash="dot", zeroline=False)
fig.add_annotation(text="a)", xref="paper", yref="paper", x=0.50, y=0.99, showarrow=False, font=dict(size=16))
fig.add_annotation(text="b)", xref="paper", yref="paper", x=0.99, y=0.99, showarrow=False, font=dict(size=16))

# Update layout
fig.update_layout(
    autosize=False, 
    barmode="overlay",
    bargap=0.1,
    legend=dict(orientation="v", y=0.94, x=0.82, bgcolor="rgba(255,255,255,0)"),
    width=650, 
    height=700, 
    boxmode='group',
    plot_bgcolor="rgba(229,236,246,0.7)",
    margin=dict(l=10, r=10, b=10, t=10), 
    font=dict(size=12, family="Arial"),
)

fig.write_image("figures/fig_runoff_contribution.svg", width=650, height=700)
fig.show()


## Plot (slides)

In [ ]:
# andeanGC data
metadata_andeangc   = pd.read_csv(Path(cfg["dir"]["andeangc"]) / "dataset/AndeanGC_metadata.csv", index_col=0)

runoff_andeangc_sim = xr.open_dataset("data/hydro/runoff_sim_andeangc.nc").q_mm_sim
runoff_andeangc_sim = runoff_andeangc_sim.resample(date="YS").sum().mean(dim="date")
runoff_andeangc_sim = runoff_andeangc_sim.to_series()

metadata_andeangc = metadata_andeangc.join(runoff_andeangc_sim.rename("runoff_annual"))
metadata_andeangc = metadata_andeangc.dropna()